# CheMeleon + Mother CatBoost

**Goal:** Use a pretrained molecular foundation model (CheMeleon) to create molecular fingerprints, then use Mother's `CatboostRegressorMother` for FreeSolv property prediction.

This example keeps the downstream model simple: CheMeleon supplies the molecular representation and Mother's CatBoost wrapper performs the regression using the same estimator API used throughout the Mother examples.

## 1 — Import Required Libraries

In [10]:
import warnings

warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from rdkit import Chem
from sklearn.metrics import root_mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

from mother.feature_generation import CheMeleonFingerprintFactory
from mother.ml import CatboostRegressorMother

## 2 — Create CheMeleon Fingerprint Factory

`CheMeleonFingerprintFactory` uses the current Chemprop API. The Chemprop extra includes the matching `cuik_molmaker` and RDKit binary dependencies.

Before running this notebook from a fresh checkout, install the locked extra from the repository root:

```bash
uv sync --extra chemprop
```

If no `checkpoint_path` is passed, Mother automatically downloads and caches the official CheMeleon foundation weights on first use. The CPU-friendly batch size below keeps memory use modest.

In [11]:
factory = CheMeleonFingerprintFactory(
    output_dim=2048,
    batch_size=128,  # lower memory footprint on CPU
    device="cpu",
)
fingerprinter = factory.get_fingerprint_generator()

print(f"✓ CheMeleon fingerprint generator ready  (output dim: {factory.output_dim})")
print("✓ Using automatic CheMeleon checkpoint provisioning (download + cache on first run)")

✓ CheMeleon fingerprint generator ready  (output dim: 2048)
✓ Using automatic CheMeleon checkpoint provisioning (download + cache on first run)


## 3 — Load the FreeSolv Dataset

[FreeSolv](https://github.com/MobleyLab/FreeSolv) contains **experimental
hydration free energies** (ΔG_hydr in kcal/mol).  We use the 50-molecule
training subset shipped with Mother's examples.


In [12]:
df = pd.read_csv("../freesolv_train.csv")
smiles = df["smiles"].tolist()
mols = [Chem.MolFromSmiles(s) for s in smiles]
targets = df["expt"].values.astype(np.float32)

print(f"Dataset: {len(df)} molecules")
df[["iupac", "smiles", "expt"]].head(10)

Dataset: 50 molecules


,iupac,smiles,expt
0,"4-methoxy-N,N-dimethyl-benzamide",CN(C)C(=O)c1ccc(cc1)OC,-11.01
1,methanesulfonyl chloride,CS(=O)(=O)Cl,-4.87
2,3-methylbut-1-ene,CC(C)C=C,1.83
3,2-ethylpyrazine,CCc1cnccn1,-5.45
4,heptan-1-ol,CCCCCCCO,-4.21
5,"3,5-dimethylphenol",Cc1cc(cc(c1)O)C,-6.27
6,"2,3-dimethylbutane",CC(C)C(C)C,2.34
7,2-methylpentan-2-ol,CCCC(C)(C)O,-3.92
8,"1,2-dimethylcyclohexane",C[C@@H]1CCCC[C@@H]1C,1.58
9,butan-2-ol,CC[C@H](C)O,-4.62


## 4 — Extract CheMeleon Fingerprints

Now we simply `fit_transform` the RDKit molecules through our transformer.


In [13]:
mols_array = np.array(mols, dtype=object)
fingerprints = fingerprinter.fit_transform(mols_array)
print(f"✓ Fingerprint matrix shape: {fingerprints.shape}")  # (50, 2048)

✓ Fingerprint matrix shape: (50, 2048)


## 5 — Train / Test Split


In [14]:
X_train, X_test, y_train, y_test = train_test_split(fingerprints, targets, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape[0]} molecules, Test: {X_test.shape[0]} molecules")
print(f"Feature dim: {X_train.shape[1]}")

Train: 40 molecules, Test: 10 molecules
Feature dim: 2048


## 6 — Train and Evaluate Mother's CatBoost Regressor

`CatboostRegressorMother` is Mother's sklearn-compatible CatBoost wrapper. It keeps CatBoost's regression behavior while providing Mother's shared estimator conventions, hyperparameter-tuning hooks, and uncertainty interface.

In [ ]:
mother_catboost_model = CatboostRegressorMother(
    iterations=500,
    max_depth=6,
    learning_rate=0.03,
    loss_function="RMSE",
    random_seed=42,
    verbose=False,
)
mother_catboost_model.fit(X_train, y_train)

y_pred = mother_catboost_model.predict(X_test)
rmse = root_mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"Mother CatBoost — RMSE: {rmse:.3f} kcal/mol, R²: {r2:.3f}")

CatBoostError: only one of the parameters depth, max_depth should be initialized.

## 7 — Visualise Predictions and Mother CatBoost Feature Importance

The parity plot gives a quick view of prediction quality. The wrapper exposes CatBoost's feature importance through the same fitted estimator.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(y_test, y_pred, alpha=0.8)
limits = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
axes[0].plot(limits, limits, "k--", linewidth=1)
axes[0].set_xlabel("Observed ΔG_hydr (kcal/mol)")
axes[0].set_ylabel("Predicted ΔG_hydr (kcal/mol)")
axes[0].set_title("Mother CatBoost parity plot")

importance = mother_catboost_model.get_feature_importance()
top_indices = np.argsort(importance)[-15:]
axes[1].barh(np.arange(len(top_indices)), importance[top_indices])
axes[1].set_yticks(np.arange(len(top_indices)))
axes[1].set_yticklabels([f"fp_{index}" for index in top_indices])
axes[1].set_xlabel("Feature importance")
axes[1].set_title("Top CheMeleon fingerprint dimensions")

plt.tight_layout()
plt.show()